# Análisis Exploratorio de Datos (EDA) - Rotación de Personal

## Objetivo
Realizar un análisis exploratorio completo de los datos de plantilla laboral y percepciones gerenciales sobre rotación de personal, identificando patrones, tendencias y generando insights accionables para la gestión estratégica de talento.

## Datasets
1. **GCP_Core_Plantilla_ActivaVSAutorizada_REP_8_Sheet1.csv** - Datos administrativos de empleados
2. **Sondeo - Perspectiva del Gerente sobre Rotación (Respuestas).xlsx** - Percepciones gerenciales

## 1. Configuración del Entorno y Carga de Librerías

In [18]:
# Librerías para manipulación de datos
import pandas as pd
import numpy as np
from datetime import datetime
import unicodedata
import re

# Librerías para visualización
import matplotlib.pyplot as plt
import seaborn as sns


# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Warnings
import warnings
warnings.filterwarnings('ignore')

print("[+] Librerías cargadas exitosamente")

[+] Librerías cargadas exitosamente


In [19]:
# Extraer el número del grado (ej. 'GLOBAL GRADE 05' -> 5)
def extraer_grado_numerico(grado):
    if pd.isna(grado):
        return None
    # Usa una expresión regular para encontrar el primer conjunto de dígitos
    match = re.search(r'(\d+)', str(grado))
    if match:
        return int(match.group(1))
    return None # Retorna None si no se encuentra un número

def limpiar_nombre_columna(texto):
    """Función para limpiar nombres de columnas"""
    # Remover acentos
    texto = unicodedata.normalize('NFD', str(texto))
    texto = ''.join(char for char in texto if unicodedata.category(char) != 'Mn')
    
    # Reemplazar espacios y caracteres especiales con guión bajo
    texto = texto.replace(' ', '_')
    texto = texto.replace('-', '_')
    texto = texto.replace(',', '')
    texto = texto.replace('(', '')
    texto = texto.replace(')', '')
    
    # Convertir a minúsculas
    texto = texto.lower()
    
    # Limpiar guiones bajos múltiples
    while '__' in texto:
        texto = texto.replace('__', '_')
    
    # Remover guión bajo al inicio/final
    texto = texto.strip('_')
    
    return texto

def reemplazar_pd_na_con_nan(df):
    """Reemplaza todos los pd.NA con np.nan"""
    df_limpio = df.copy()
    
    # Reemplazar pd.NA con np.nan en todo el DataFrame
    df_limpio = df_limpio.replace({pd.NA: np.nan})
    
    return df_limpio


# Convertir columnas específicas
def convertir_columnas_especificas(df, columnas):
    """Convierte columnas específicas a entero"""
    
    for col in columnas:
        if col in df.columns:
            try:
                # Convertir a numérico y luego a entero
                df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
                print(f"[+] Convertido: {col}")
                
            except Exception as e:
                print(f"[-] Error en {col}: {e}")
    
    return df

## Carga y Exploración Inicial de Datos

### Dataset de Plantilla (Datos Administrativos)

In [20]:
# Cargar dataset de plantilla
df_plantilla = pd.read_csv('../data/GCP_Core_Plantilla_ActivaVSAutorizada_REP_8_Sheet1.csv', 
                           low_memory=False)
df_plantilla = df_plantilla.iloc[:, :-4]
df_plantilla.columns = df_plantilla.columns.str.strip()  # Eliminar espacios
df_plantilla.columns = [limpiar_nombre_columna(col) for col in df_plantilla.columns]
# Aplicar conversión
columnas_enteras = [
    'centro',
    'codigo_de_puesto',
    'codigo_de_posicion',
    'codigo_de_posicion_principal',
    'plantilla_autorizada',
    'plantilla_activa',
    'vacantes',
    'numero_trabajador',
    'numero_de_persona', 
    'numero_de_trabajador', 
    'numero_de_persona_del_manager',	
    'numero_de_trabajador_del_manage'
]
df_platilla = convertir_columnas_especificas(df_plantilla, columnas_enteras)
df_plantilla = reemplazar_pd_na_con_nan(df_plantilla)

[+] Convertido: centro
[+] Convertido: codigo_de_puesto
[+] Convertido: codigo_de_posicion
[+] Convertido: codigo_de_posicion_principal
[+] Convertido: plantilla_autorizada
[+] Convertido: plantilla_activa
[+] Convertido: vacantes
[+] Convertido: numero_de_persona
[+] Convertido: numero_de_trabajador
[+] Convertido: numero_de_persona_del_manager


In [21]:
# Primeras filas
print(f"Dimensiones del dataset: {df_plantilla.shape}")
print(f"Filas: {df_plantilla.shape[0]:,}")
print(f"Columnas: {df_plantilla.shape[1]}")
df_plantilla.sample(5)

Dimensiones del dataset: (199599, 41)
Filas: 199,599
Columnas: 41


,fecha_de_inicio_de_vigencia,fecha_de_cierre,estado,estado_contratacion,posicion_presupuestada,importe,centro_de_costos,centro,ubicacion,region,ciudad,empleador_legal,unidad_de_negocio,division,departamento,codigo_de_puesto,nombre_de_puesto,familia_de_puesto,funcion_del_puesto,codigo_de_posicion,nombre_de_posicion,indefinido_o_temporal,tipo_de_presupuesto,categoria_de_asignacion,plantilla_autorizada,plantilla_activa,vacantes,grupo_salarial,rol_de_contribucion,numero_de_persona,numero_de_trabajador,nombre_del_colaborador,fecha_de_antiguedad_grupo,fecha_de_contratacion_empleador_legal_actual,fecha_de_entrada_en_posicion_actual,tipo_de_trabajador,codigo_de_posicion_principal,nombre_de_posicion_principal,numero_de_persona_del_manager,numero_de_trabajador_del_manager,nombre_del_manager
97012,09/05/2025,NaN,Activa,Aprobada,Sí,$1.00,007 - COBRANZAS,511801,MX-03-MOR-197 - C13-CUAUTLA-197,IXTAPALUCA,CUAUTLA MORELOS,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 OPERACION DE COBRANZA IXTP 02,71,SUPERVISOR DE COBRANZA,Soporte al Negocio,SUPERVISOR,12221,SUPERVISOR DE COBRANZA,Indefinido,Operativo,Operacion,1,1,0,GLOBAL GRADE 05,U2,100121835,90483423,Enrique Quintero González,03/09/2025,03/09/2025,03/09/2025,Colaborador,12194,GERENTE DE OPERACION COBRANZA,95830634,95830634.00,Ivan Canales Rivera
51344,09/05/2025,NaN,Activa,Aprobada,Sí,$1.00,001 - TDAS MUEBLES,122501,MX-01-VER-1121 - T18-LA FLORIDA-1121,VERACRUZ,OLUTA VER,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 TIENDA MUEBLES 1225 LA FLORIDA,21,CAJERO MULTIFUNCIONAL,Soporte al Negocio,CAJERO,86984,CAJERO MULTIFUNCIONAL,Indefinido,Operativo,Operacion,1,1,0,GLOBAL GRADE 05,U2,90358641,90358641,Elizabeth Jimenez Hernandez,26/01/2024,26/01/2024,05/02/2024,Colaborador,86980,GERENTE DE TIENDA MUEBLES,96182326,96182326.00,Missael Dominguez Martinez
26855,25/06/2025,NaN,Activa,Aprobada,Sí,$1.00,005 - TDAS CANADAS,70402,MX-01-JAL-653 - T17-LOPEZ PAJAR-653,GUADALAJARA II,"TONALA,JALISCO",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 TIENDA ROPA 0704 LOPEZ PAJAR,453,ASESOR DE TELEFONIA LIBRE,Operativo,ASESOR,150514,ASESOR DE TELEFONIA LIBRE,Indefinido,Operativo,Operacion,1,1,0,GLOBAL GRADE 05,W2,90354459,90354459,Cesar Ezequiel Camacho Haro,08/11/2023,08/11/2023,05/02/2024,Colaborador,69618,GERENTE DE TIENDA ROPA,98590952,98590952.00,Miguel Angel Marquez Cornejo
163075,09/05/2025,NaN,Inactiva,Congelada,No,$0.00,109 - SALE VALE,750660,MX-03-MICH-210 - C16-PATZCUARO-210,TOLUCA,"PATZCUARO, MICH",NaN,BU_SASACV,VICEPRESIDENCIA DE SERVICIOS FINANCIEROS GC,06 SUCURSAL PTZC,72,PROMOTOR DE CREDITO,Soporte al Negocio,PROMOTOR,145176,PROMOTOR DE CREDITO,Indefinido,Operativo,Operacion,1,0,1,GLOBAL GRADE 05,U1,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,145171,GERENTE DE SUCURSAL,90313907,90313907.00,Juan Paulo Olguin Rojas
28027,23/06/2025,NaN,Activa,Congelada,Sí,$1.00,005 - TDAS CANADAS,61602,MX-01-MEX-580 - T26-CHIMALHUACAN-580,TEXCOCO,CHIMALHUACAN EM,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 TIENDA ROPA 0616 CHIMALHUACAN,453,ASESOR DE TELEFONIA LIBRE,Operativo,ASESOR,68284,ASESOR DE TELEFONIA LIBRE,Indefinido,Operativo,Operacion,1,1,0,GLOBAL GRADE 05,W2,98717863,98717863,Laura Denis Lerma Casimiro,19/10/2022,19/10/2022,04/11/2023,Colaborador,68283,GERENTE DE TIENDA ROPA,95007172,95007172.00,Leticia Santamaria Guzman


In [53]:
def calcular_subordinados_y_posiciones(df_plantilla):
    """
    Calcula subordinados y posiciones a cargo para cada manager
    Incluye: lista de subordinados, moda del grupo salarial, contrataciones congeladas, promedio de antigüedad
    """
    
    print("Calculando subordinados y posiciones a cargo...")
        
    # Crear DataFrame de trabajo
    df_trabajo = df_plantilla.copy()
    
    
    print(f"Registros a analizar: {len(df_trabajo)}")
    
    # Identificar todas las personas únicas (tanto empleados como managers)
    todas_las_personas = set()
    
    # Agregar todos los empleados
    empleados = df_trabajo['numero_de_persona'].dropna().unique()
    todas_las_personas.update(empleados)
    
    # Agregar todos los managers
    managers = df_trabajo['numero_de_persona_del_manager'].dropna().unique()
    todas_las_personas.update(managers)
    
    print(f"Total de personas únicas identificadas: {len(todas_las_personas)}")
    
    # Crear DataFrame resultado
    resultados = []
    
    for persona_id in todas_las_personas:
        try:
            persona_id_num = int(persona_id)
            
            # Obtener información básica de la persona
            info_persona = df_trabajo[df_trabajo['numero_de_persona'] == persona_id_num]
            
            if len(info_persona) > 0:
                nombre_persona = info_persona['nombre_del_colaborador'].iloc[0]
                estado_persona = info_persona['estado'].iloc[0]
                numero_trabajador = info_persona['numero_de_trabajador'].iloc[0]

            else:
                # Si no está como empleado, podría ser solo manager
                print(" Persona no encontrada como empleado, buscando como manager...")
                info_como_manager = df_trabajo[df_trabajo['numero_de_persona_del_manager'] == persona_id_num]
                if len(info_como_manager) > 0:
                
                    nombre_persona = info_como_manager['nombre_del_manager'].iloc[0]
                    estado_persona = "Manager sin registro directo"
                    numero_trabajador = info_como_manager['numero_de_trabajador_del_manager'].iloc[0] if 'numero_de_trabajador_del_manager' in info_como_manager.columns else None
                else:
                    nombre_persona = f"Persona ID {persona_id_num}"
                    estado_persona = "Sin información"
                    numero_trabajador = None
            
            # Obtener TODOS los subordinados de esta persona (sin filtrar por estado)
            todos_subordinados = df_trabajo[df_trabajo['numero_de_persona_del_manager'] == persona_id_num]
            
            tiene_subordinados = df_trabajo[df_trabajo['numero_de_persona_del_manager'] == persona_id_num].shape[0] > 0

            total_posiciones_a_cargo = len(todos_subordinados)
            total_subordinados_activos = len(todos_subordinados[todos_subordinados['plantilla_activa'] == 1])
            total_posiciones_vacantes = len(todos_subordinados[todos_subordinados['vacantes'] == 0])

            # Clasificar tipo de rol
            if total_posiciones_a_cargo == 0:
                tipo_rol = "Sin subordinados"
            elif total_subordinados_activos > 0 and total_posiciones_vacantes == 0:
                tipo_rol = "Manager con equipo completo"
            elif total_subordinados_activos == 0 and total_posiciones_vacantes > 0:
                tipo_rol = "Manager con solo vacantes"
            else:
                tipo_rol = "Manager con equipo y vacantes"
            
            # Lista de números de trabajador de subordinados
            lista_subordinados = []
            if len(todos_subordinados) > 0:
                subordinados_validos = todos_subordinados['numero_de_trabajador'].dropna()
                lista_subordinados = subordinados_validos.tolist()
            
            
            
            # grupo salarial de subordinados
            moda_grupo_salarial = None
            grupos_salariales_numeric = None
            grupos_salariales = None
            lista_centros_subordinados = None
            promedio_grupo_salarial = None
            std_grupo_salarial = None

            if 'grupo_salarial' in df_trabajo.columns and len(todos_subordinados) > 0:
                grupos_salariales = todos_subordinados['grupo_salarial'].dropna()
                lista_centros_subordinados = todos_subordinados['centro'].dropna().unique().tolist()

                # 1. Verificar si hay grupos salariales después de dropear NaNs
                if not grupos_salariales.empty: 
                    try:
                        # APLICAR LA FUNCIÓN A CADA ELEMENTO de la Serie y quitar NaNs
                        grupos_salariales_numeric = grupos_salariales.apply(extraer_grado_numerico).dropna()
                        
                        # 2. Verificar si hay valores numéricos válidos después de la transformación
                        if not grupos_salariales_numeric.empty: # <--- Usar .empty para Series
                            
                            promedio_grupo_salarial = grupos_salariales_numeric.mean()
                            std_grupo_salarial = grupos_salariales_numeric.std()
                            # Calcular la moda y asegurarse de que no esté vacía
                            moda_series = grupos_salariales_numeric.mode()
                            if not moda_series.empty:
                                moda_grupo_salarial = moda_series.iloc[0]

                    except Exception as e:
                        # Captura errores en el cálculo numérico (ej. si quedan no-numéricos)
                        print(f"Error calculando moda para persona {persona_id_num}: {e}")
                        moda_grupo_salarial = None
                        promedio_grupo_salarial = None
                        std_grupo_salarial = None
                        grupos_salariales_numeric = None
            
            # Promedio y std de antigüedad de subordinados
            promedio_antiguedad_subordinados = None
            std_antiguedad_equipo = None  
            if 'fecha_de_antiguedad_grupo' in df_trabajo.columns and len(todos_subordinados) > 0:
                try:
                    # Convertir fechas y calcular antigüedad
                    fechas_antiguedad = pd.to_datetime(todos_subordinados['fecha_de_antiguedad_grupo'], errors='coerce')
                    fechas_validas = fechas_antiguedad.dropna()
                    
                    if len(fechas_validas) > 0:
                        fecha_actual = pd.Timestamp.now()
                        antiguedades_dias = (fecha_actual - fechas_validas).dt.days
                        
                        # Filtrar antigüedades válidas (positivas y razonables)
                        antiguedades_validas = antiguedades_dias[(antiguedades_dias >= 0) & (antiguedades_dias <= 50*365)]
                        
                        if len(antiguedades_validas) > 0:
                            promedio_antiguedad_subordinados = antiguedades_validas.mean() / 365.25  # Convertir a años
                            std_antiguedad_equipo = antiguedades_validas.std() / 365.25  # Convertir a años


                except Exception as e:
                    print(f"Error calculando antigüedad promedio para persona {persona_id_num}: {e}")
                    promedio_antiguedad_subordinados = None
            
            
            # Agregar resultado completo
            resultado = {
                'numero_de_persona': persona_id_num,
                'numero_de_trabajador': numero_trabajador,
                'nombre_de_colaborador': nombre_persona,
                'tipo_rol': tipo_rol,
                'tiene_subordinados': tiene_subordinados,
                'fecha_antiguedad_grupo': info_persona['fecha_de_antiguedad_grupo'].iloc[0] if len(info_persona) > 0 else None,
                'centro_de_costos': info_persona['centro_de_costos'].iloc[0] if len(info_persona) > 0 else None,
                'centro': info_persona['centro'].iloc[0] if len(info_persona) > 0 else None,
                'ubicacion': info_persona['ubicacion'].iloc[0] if len(info_persona) > 0 else None,
                'region': info_persona['region'].iloc[0] if len(info_persona) > 0 else None,
                'ciudad': info_persona['ciudad'].iloc[0] if len(info_persona) > 0 else None,
                'empleador_legal':  info_persona['empleador_legal'].iloc[0] if len(info_persona) > 0 else None,
                'unidad_de_negocio': info_persona['unidad_de_negocio'].iloc[0] if len(info_persona) > 0 else None,
                'division': info_persona['division'].iloc[0] if len(info_persona) > 0 else None,
                'departamento': info_persona['departamento'].iloc[0] if len(info_persona) > 0 else None,
                'codigo_de_puesto': info_persona['codigo_de_puesto'].iloc[0] if len(info_persona) > 0 else None,
                'nombre_de_puesto': info_persona['nombre_de_puesto'].iloc[0] if len(info_persona) > 0 else None,
                'familia_de_puesto': info_persona['familia_de_puesto'].iloc[0] if len(info_persona) > 0 else None,
                'funcion_del_puesto': info_persona['funcion_del_puesto'].iloc[0] if len(info_persona) > 0 else None,
                'codigo_de_posicion': info_persona['codigo_de_posicion'].iloc[0] if len(info_persona) > 0 else None,
                'nombre_de_posicion': info_persona['nombre_de_posicion'].iloc[0] if len(info_persona) > 0 else None,
                'total_plantilla_autorizada': len(todos_subordinados[todos_subordinados['plantilla_autorizada'] == 1]) if 'plantilla_autorizada' in df_trabajo.columns else None,
                'total_plantilla_activa': len(todos_subordinados[todos_subordinados['plantilla_activa'] == 1]) if 'plantilla_activa' in df_trabajo.columns else None,
                'posiciones_vacantes': len(todos_subordinados[todos_subordinados['vacantes'] == 1]) if 'vacantes' in df_trabajo.columns else None,
                'total_contrataciones_congeladas':len(todos_subordinados[todos_subordinados['estado_contratacion'] == 'Congelada']) if 'estado_contratacion' in df_trabajo.columns else None,
                'total_contrataciones_aprobadas': len(todos_subordinados[todos_subordinados['estado_contratacion'] == 'Aprobada']) if 'estado_contratacion' in df_trabajo.columns else None,
                'total_posiciones_presupuestadas': len(todos_subordinados[todos_subordinados['posicion_presupuestada'] == 'Sí']) if 'posicion_presupuestada' in df_trabajo.columns else None,
                'total_indefinidos_a_cargo': len(todos_subordinados[todos_subordinados['indefinido_o_temporal'] == 'Indefinido']) if 'indefinido_o_temporal' in df_trabajo.columns else None,
                'lista_subordinados_trabajador': lista_subordinados,
                'grupos_salariales': grupos_salariales_numeric.tolist() if grupos_salariales_numeric is not None else None,
                'lista_centros_subordinados': lista_centros_subordinados,
                'promedio_grupo_salarial': promedio_grupo_salarial,
                'std_grupo_salarial': std_grupo_salarial,
                'moda_grupo_salarial': moda_grupo_salarial,
                'promedio_grupo_salarial': promedio_grupo_salarial,
                'std_grupo_salarial': std_grupo_salarial,
                'promedio_antiguedad_subordinados_anos': promedio_antiguedad_subordinados,
                'std_antiguedad_equipo': std_antiguedad_equipo
            }
            resultados.append(resultado)
            
            
        except Exception as e:
            print(f"[-] Error procesando persona {persona_id}: {e}")
            continue
    
    # Crear DataFrame resultado
    df_resultado = pd.DataFrame(resultados)
    
    # Ordenar por total de posiciones a cargo (descendente)
    df_resultado = df_resultado.sort_values('total_plantilla_autorizada', ascending=False)
    
    print(f"[+] Análisis completado para {len(df_resultado)} personas")
    
    # Mostrar estadísticas
    if len(df_resultado) > 0:
        print(f"\n ESTADÍSTICAS:")
        
        
        # Estadísticas de grupo salarial
        if 'grupo_salarial' in df_plantilla.columns:
            personas_con_moda = df_resultado[df_resultado['moda_grupo_salarial'].notna()]
            print(f" Managers con moda de grupo salarial calculada: {len(personas_con_moda)}")
            if len(personas_con_moda) > 0:
                modas_comunes = personas_con_moda['moda_grupo_salarial'].value_counts().head(3)
                print(f" Grupos salariales más comunes como moda: {modas_comunes.to_dict()}")
        
        
        # Estadísticas de antigüedad
        if 'fecha_de_antiguedad_grupo' in df_plantilla.columns:
            personas_con_antiguedad = df_resultado[df_resultado['promedio_antiguedad_subordinados_anos'].notna()]
            if len(personas_con_antiguedad) > 0:
                print(f" Managers con promedio de antigüedad calculado: {len(personas_con_antiguedad)}")
                print(f" Antigüedad promedio general: {personas_con_antiguedad['promedio_antiguedad_subordinados_anos'].mean():.2f} años")
    
    return df_resultado

In [ ]:
%%time
df_jerarquia = calcular_subordinados_y_posiciones(df_plantilla)
df_jerarquia
df_jerarquia.head()

Calculando subordinados y posiciones a cargo...
Registros a analizar: 199599
Total de personas únicas identificadas: 122854
[+] Análisis completado para 122854 personas

 ESTADÍSTICAS:
 Managers con moda de grupo salarial calculada: 10381
 Grupos salariales más comunes como moda: {5.0: 5570, 10.0: 1445, 11.0: 755}
 Managers con promedio de antigüedad calculado: 9824
 Antigüedad promedio general: 4.55 años
CPU times: total: 6min 2s
Wall time: 12min 16s


,numero_de_persona,numero_de_trabajador,nombre_de_colaborador,tipo_rol,tiene_subordinados,fecha_antiguedad_grupo,centro_de_costos,centro,ubicacion,region,ciudad,empleador_legal,unidad_de_negocio,division,departamento,codigo_de_puesto,nombre_de_puesto,familia_de_puesto,funcion_del_puesto,codigo_de_posicion,nombre_de_posicion,total_plantilla_autorizada,total_plantilla_activa,posiciones_vacantes,total_contrataciones_congeladas,total_contrataciones_aprobadas,total_posiciones_presupuestadas,total_indefinidos_a_cargo,lista_subordinados_trabajador,grupos_salariales,lista_centros_subordinados,promedio_grupo_salarial,std_grupo_salarial,moda_grupo_salarial,promedio_antiguedad_subordinados_anos,std_antiguedad_equipo
31833,90021474,90021474,Christian Lucario Balderas,Manager con equipo y vacantes,True,02/04/2019,092 - ZONA CEDIS MUEBLES IMPORTACION,201854,MX-02-MEX-93 - T11-TECAMAC/ BODEGA VANCOUVER-93,TECAMAC,"TECAMAC, EDO ME",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 EMBARQUES TCMC 03,1053,GERENTE DE EMBARQUES,Manager,GERENTE,33206,GERENTE DE EMBARQUES,238,204,34,8,230,223,77,"[90477306, 90478010, 90477775, 90383218, 90478...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...",[201854],4.95,0.44,5.00,1.47,2.27
109843,94870675,94870675,Maria Teresa Ochoa Gomez,Manager con equipo y vacantes,True,18/11/2022,072 - TRASLADO,218229,MX-05-MEX-40 - R27-OFICINA STAFF-COPPEL-40,TEXCOCO,TEXCOCO EDO MEX,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 MODULO TIENDAS TXCC,39,GERENTE DE AREA CEDIS,Manager,GERENTE,134147,GERENTE DE AREA CEDIS,187,138,49,43,144,185,104,"[90437075, 90473926, 90361942, 90361529, 90474...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[218234, 218229]",5.03,0.37,5.00,1.89,1.99
16021,92336515,92336515,Aaron Aguilar Herrera,Manager con equipo y vacantes,True,24/09/2007,079 - BODEGA ROPA STAFF,218228,MX-05-MEX-40 - R27-OFICINA STAFF-COPPEL-40,TEXCOCO,TEXCOCO EDO MEX,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 RESGUARDO TXCC,39,GERENTE DE AREA CEDIS,Manager,GERENTE,134053,GERENTE DE AREA CEDIS,161,106,55,0,161,127,92,"[90484232, 90483427, 90483469, 90482950, 90484...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[218234, 218228]",4.22,0.54,4.00,1.59,1.16
121084,92268196,92268196,Camilo Leonel Espinosa Barrios,Manager con equipo y vacantes,True,19/06/2008,065 - DESCARGA DISTRIBUCION,153207,MX-04-MEX-74 - R20-TOLC PERSONAL-74,TOLUCA,"TOLUCA, EDO MEX",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 DESCARGA TOLC,1052,GERENTE DE DESCARGA,Manager,GERENTE,19999,GERENTE DE DESCARGA,161,128,33,27,134,155,90,"[90479699, 90368182, 90481489, 90479716, 90425...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...",[153207],4.99,0.40,5.00,1.56,2.16
25648,90156373,90156373,Daniel Martin Lopez Gonzalez,Manager con equipo y vacantes,True,26/02/2021,065 - DESCARGA DISTRIBUCION,169905,MX-01-PUE-989 - T08-AV. MEXICO-989,PUEBLA,"CUAUTLANCINGO, PUE",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 DESCARGA PBLA 02,1052,GERENTE DE DESCARGA,Manager,GERENTE,22291,GERENTE DE DESCARGA,144,93,51,32,112,132,78,"[90479328, 90482889, 90395636, 90483317, 90333...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[169905, 201212]",5.12,0.61,5.00,1.66,2.33


In [98]:
df_jerarquia['posiciones_vacantes_activas'] = df_jerarquia['posiciones_vacantes'] - df_jerarquia['total_contrataciones_congeladas']

In [130]:
df_jerarquia['len_lista_subordinados_trabajador'] = df_jerarquia['lista_subordinados_trabajador'].apply(lambda x: len(x) if isinstance(x, list) else 0)


In [131]:
#gerente = 'Ulises Ruiz Aguilar'
#gerente = 'Lehi Garcia Serrano'
#gerente = 'Misael Dominguez Cruz'
#gerente = 'Jorge Roy Ruiz Alcantara'
#gerente = 'Jose Trinidad Benitez Angulo'
gerente = 'Guadalupe Berenice Baez Hernandez'
df_jerarquia[df_jerarquia['nombre_de_colaborador'] == gerente][
    ['nombre_de_colaborador', 'total_plantilla_autorizada', 'total_plantilla_activa', 'posiciones_vacantes', 
    'total_contrataciones_congeladas', 'posiciones_vacantes_activas', 'total_contrataciones_aprobadas', 
    'total_posiciones_presupuestadas', 'len_lista_subordinados_trabajador']]

,nombre_de_colaborador,total_plantilla_autorizada,total_plantilla_activa,posiciones_vacantes,total_contrataciones_congeladas,posiciones_vacantes_activas,total_contrataciones_aprobadas,total_posiciones_presupuestadas,len_lista_subordinados_trabajador
85801,Guadalupe Berenice Baez Hernandez,63,38,25,19,6,44,38,38


In [136]:
lista_centros = df_jerarquia[df_jerarquia['nombre_de_colaborador'] == gerente][['lista_centros_subordinados']].to_dict(orient='records')[0]['lista_centros_subordinados']

In [134]:
df_plantilla[df_plantilla['nombre_del_manager'] == gerente][[
    'estado', 'estado_contratacion', 'centro',
    'posicion_presupuestada', 'plantilla_autorizada', 
    'plantilla_activa', 'vacantes', 'numero_de_trabajador', 'nombre_de_posicion' ,'nombre_del_colaborador', 'nombre_del_manager']]

,estado,estado_contratacion,centro,posicion_presupuestada,plantilla_autorizada,plantilla_activa,vacantes,numero_de_trabajador,nombre_de_posicion,nombre_del_colaborador,nombre_del_manager
689,Inactiva,Congelada,315401,No,1,0,1,<NA>,GERENTE SUPLENTE,NaN,Guadalupe Berenice Baez Hernandez
12464,Activa,Congelada,300267,No,1,1,0,97284131,AUXILIAR DE INVENTARIOS,Jose Armando Romero Lopez,Guadalupe Berenice Baez Hernandez
12582,Activa,Congelada,300267,Sí,1,1,0,97608751,AUXILIAR DE INVENTARIOS,Fernanda Belen Ruiz Gonzalez,Guadalupe Berenice Baez Hernandez
12583,Activa,Congelada,300267,Sí,1,1,0,92030611,AUXILIAR DE INVENTARIOS,Estela Garcia Garcia,Guadalupe Berenice Baez Hernandez
12584,Activa,Congelada,300267,Sí,1,1,0,90101008,AUXILIAR DE INVENTARIOS,Abel Calderon Rodriguez,Guadalupe Berenice Baez Hernandez
13405,Activa,Aprobada,121410,Sí,1,1,0,95426574,GERENTE SUPLENTE,Angel Fernando Correa Campiran,Guadalupe Berenice Baez Hernandez
20477,Activa,Aprobada,300266,No,1,1,0,96863871,AUXILIAR DE INVENTARIOS,Alejandro Neri Agonizantes,Guadalupe Berenice Baez Hernandez
33296,Inactiva,Congelada,33210,No,1,0,1,<NA>,GERENTE GENERAL DE TIENDA,NaN,Guadalupe Berenice Baez Hernandez
40628,Activa,Congelada,300265,No,1,1,0,98293818,AUXILIAR DE INVENTARIOS,Adriana Erika Perez Fonseca,Guadalupe Berenice Baez Hernandez
40629,Activa,Congelada,300265,No,1,1,0,96159324,AUXILIAR DE INVENTARIOS,Yoselin Rios Serrano,Guadalupe Berenice Baez Hernandez


In [141]:
(df_plantilla[df_plantilla['centro'].isin(lista_centros)].sort_values('nombre_del_manager')[[
    'estado', 'estado_contratacion', 'centro',
    'posicion_presupuestada', 'plantilla_autorizada', 
    'plantilla_activa', 'vacantes', 'numero_de_trabajador', 'nombre_de_posicion' ,'nombre_del_colaborador', 'nombre_del_manager']]).to_csv('../data/reporte_gerente_guadalupe_baez_hernandez.csv', index=False)

In [ ]:
df_plantilla[df_plantilla['centro'] == 321273]

,fecha_de_inicio_de_vigencia,fecha_de_cierre,estado,estado_contratacion,posicion_presupuestada,importe,centro_de_costos,centro,ubicacion,region,ciudad,empleador_legal,unidad_de_negocio,division,departamento,codigo_de_puesto,nombre_de_puesto,familia_de_puesto,funcion_del_puesto,codigo_de_posicion,nombre_de_posicion,indefinido_o_temporal,tipo_de_presupuesto,categoria_de_asignacion,plantilla_autorizada,plantilla_activa,vacantes,grupo_salarial,rol_de_contribucion,numero_de_persona,numero_de_trabajador,nombre_del_colaborador,fecha_de_antiguedad_grupo,fecha_de_contratacion_empleador_legal_actual,fecha_de_entrada_en_posicion_actual,tipo_de_trabajador,codigo_de_posicion_principal,nombre_de_posicion_principal,numero_de_persona_del_manager,numero_de_trabajador_del_manager,nombre_del_manager
2152,01/09/2025,NaN,Activa,Aprobada,Sí,$1.00,227 - INVESTIGACION COPPEL,321273,MX-05-CDMX-24 - R05-CORPORATIVO INSURGENTES 55...,AZCAPOTZALCO,MIGUEL HGO D.F.,COPPEL SA DE CV,BU_CSACV,DIRECCION DE ESTRATEGIA Y CRECIMIENTO GC,06 INTEGRACION Y DESPLIEGUE,4886,INGENIERO EN ML,Profesionista,ESPECIALISTA,114408,INGENIERO EN ML,Indefinido,Staff,Staff,1,1,0,GLOBAL GRADE 11,P3,90242894,90242894,Christian Gustavo Martinez Ramirez,17/06/2022,17/06/2022,04/11/2023,Colaborador,113128,GERENTE DE INTEGRACION Y DESPLIEGUE,90160643,90160643.00,Jorge Roy Ruiz Alcantara
2153,01/09/2025,NaN,Activa,Aprobada,Sí,$1.00,227 - INVESTIGACION COPPEL,321273,MX-05-CDMX-24 - R05-CORPORATIVO INSURGENTES 55...,AZCAPOTZALCO,MIGUEL HGO D.F.,COPPEL SA DE CV,BU_CSACV,DIRECCION DE ESTRATEGIA Y CRECIMIENTO GC,06 INTEGRACION Y DESPLIEGUE,4886,INGENIERO EN ML,Profesionista,ESPECIALISTA,114409,INGENIERO EN ML,Indefinido,Staff,Staff,1,1,0,GLOBAL GRADE 11,P3,100066294,90437483,Carlos Cuauhtemoc Gutierrez Salazar,02/12/2024,02/12/2024,02/12/2024,Colaborador,113128,GERENTE DE INTEGRACION Y DESPLIEGUE,90160643,90160643.00,Jorge Roy Ruiz Alcantara
2154,01/09/2025,NaN,Activa,Aprobada,Sí,$1.00,227 - INVESTIGACION COPPEL,321273,MX-05-CDMX-24 - R05-CORPORATIVO INSURGENTES 55...,AZCAPOTZALCO,MIGUEL HGO D.F.,COPPEL SA DE CV,BU_CSACV,DIRECCION DE ESTRATEGIA Y CRECIMIENTO GC,06 INTEGRACION Y DESPLIEGUE,4886,INGENIERO EN ML,Profesionista,ESPECIALISTA,114410,INGENIERO EN ML,Indefinido,Staff,Staff,1,1,0,GLOBAL GRADE 11,P3,98783831,98783831,Alejandro Agustin Ahedo Gonzalez,25/09/2018,25/09/2018,04/11/2023,Directivo,113128,GERENTE DE INTEGRACION Y DESPLIEGUE,90160643,90160643.00,Jorge Roy Ruiz Alcantara
21894,15/07/2025,NaN,Activa,Aprobada,Sí,$1.00,227 - INVESTIGACION COPPEL,321273,MX-05-CDMX-24 - R05-CORPORATIVO INSURGENTES 55...,AZCAPOTZALCO,MIGUEL HGO D.F.,COPPEL SA DE CV,BU_CSACV,DIRECCION DE ESTRATEGIA Y CRECIMIENTO GC,06 INVEST TECNOLOGICA Y VISUALIZACION,1389,GERENTE DE INVESTIGACION INTELIGENCIA ARTIFICIAL,Manager,GERENTE,114407,GERENTE DE INVESTIGACION INTELIGENCIA ARTIFICIAL,Indefinido,Staff,Staff,1,1,0,GLOBAL GRADE 13,M2,100109090,90471883,Allan Eduardo Rosas Garcia,17/07/2025,17/07/2025,17/07/2025,Colaborador,114368,GERENTE SR INVESTIGACION IA Y ANALITICA,90030806,90030806.00,Jose Eduardo Lozas Galicia


In [59]:
## anios antiguedad
df_jerarquia['antiguedad_anois'] = df_jerarquia['fecha_antiguedad_grupo'].apply(
    lambda x: (pd.Timestamp.now() - pd.to_datetime(x)).days / 365.25 if pd.notna(x) else None
)  
df_jerarquia.head()

,numero_de_persona,numero_de_trabajador,nombre_de_colaborador,tipo_rol,tiene_subordinados,fecha_antiguedad_grupo,centro_de_costos,centro,ubicacion,region,ciudad,empleador_legal,unidad_de_negocio,division,departamento,codigo_de_puesto,nombre_de_puesto,familia_de_puesto,funcion_del_puesto,codigo_de_posicion,nombre_de_posicion,total_plantilla_autorizada,total_plantilla_activa,posiciones_vacantes,total_contrataciones_congeladas,total_contrataciones_aprobadas,total_posiciones_presupuestadas,total_indefinidos_a_cargo,lista_subordinados_trabajador,grupos_salariales,lista_centros_subordinados,promedio_grupo_salarial,std_grupo_salarial,moda_grupo_salarial,promedio_antiguedad_subordinados_anos,std_antiguedad_equipo,antiguedad_anois
31833,90021474,90021474,Christian Lucario Balderas,Manager con equipo y vacantes,True,02/04/2019,092 - ZONA CEDIS MUEBLES IMPORTACION,201854,MX-02-MEX-93 - T11-TECAMAC/ BODEGA VANCOUVER-93,TECAMAC,"TECAMAC, EDO ME",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 EMBARQUES TCMC 03,1053,GERENTE DE EMBARQUES,Manager,GERENTE,33206,GERENTE DE EMBARQUES,238,204,34,8,230,223,77,"[90477306, 90478010, 90477775, 90383218, 90478...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...",[201854],4.95,0.44,5.00,1.47,2.27,6.72
109843,94870675,94870675,Maria Teresa Ochoa Gomez,Manager con equipo y vacantes,True,18/11/2022,072 - TRASLADO,218229,MX-05-MEX-40 - R27-OFICINA STAFF-COPPEL-40,TEXCOCO,TEXCOCO EDO MEX,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 MODULO TIENDAS TXCC,39,GERENTE DE AREA CEDIS,Manager,GERENTE,134147,GERENTE DE AREA CEDIS,187,138,49,43,144,185,104,"[90437075, 90473926, 90361942, 90361529, 90474...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[218234, 218229]",5.03,0.37,5.00,1.89,1.99,2.93
16021,92336515,92336515,Aaron Aguilar Herrera,Manager con equipo y vacantes,True,24/09/2007,079 - BODEGA ROPA STAFF,218228,MX-05-MEX-40 - R27-OFICINA STAFF-COPPEL-40,TEXCOCO,TEXCOCO EDO MEX,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 RESGUARDO TXCC,39,GERENTE DE AREA CEDIS,Manager,GERENTE,134053,GERENTE DE AREA CEDIS,161,106,55,0,161,127,92,"[90484232, 90483427, 90483469, 90482950, 90484...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[218234, 218228]",4.22,0.54,4.00,1.59,1.16,18.08
121084,92268196,92268196,Camilo Leonel Espinosa Barrios,Manager con equipo y vacantes,True,19/06/2008,065 - DESCARGA DISTRIBUCION,153207,MX-04-MEX-74 - R20-TOLC PERSONAL-74,TOLUCA,"TOLUCA, EDO MEX",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 DESCARGA TOLC,1052,GERENTE DE DESCARGA,Manager,GERENTE,19999,GERENTE DE DESCARGA,161,128,33,27,134,155,90,"[90479699, 90368182, 90481489, 90479716, 90425...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...",[153207],4.99,0.40,5.00,1.56,2.16,17.34
25648,90156373,90156373,Daniel Martin Lopez Gonzalez,Manager con equipo y vacantes,True,26/02/2021,065 - DESCARGA DISTRIBUCION,169905,MX-01-PUE-989 - T08-AV. MEXICO-989,PUEBLA,"CUAUTLANCINGO, PUE",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 DESCARGA PBLA 02,1052,GERENTE DE DESCARGA,Manager,GERENTE,22291,GERENTE DE DESCARGA,144,93,51,32,112,132,78,"[90479328, 90482889, 90395636, 90483317, 90333...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[169905, 201212]",5.12,0.61,5.00,1.66,2.33,4.65


In [60]:
## Numero de centros a cargo
df_jerarquia['numero_centros_a_cargo'] = df_jerarquia['lista_centros_subordinados'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)
df_jerarquia.head()

,numero_de_persona,numero_de_trabajador,nombre_de_colaborador,tipo_rol,tiene_subordinados,fecha_antiguedad_grupo,centro_de_costos,centro,ubicacion,region,ciudad,empleador_legal,unidad_de_negocio,division,departamento,codigo_de_puesto,nombre_de_puesto,familia_de_puesto,funcion_del_puesto,codigo_de_posicion,nombre_de_posicion,total_plantilla_autorizada,total_plantilla_activa,posiciones_vacantes,total_contrataciones_congeladas,total_contrataciones_aprobadas,total_posiciones_presupuestadas,total_indefinidos_a_cargo,lista_subordinados_trabajador,grupos_salariales,lista_centros_subordinados,promedio_grupo_salarial,std_grupo_salarial,moda_grupo_salarial,promedio_antiguedad_subordinados_anos,std_antiguedad_equipo,antiguedad_anois,numero_centros_a_cargo
31833,90021474,90021474,Christian Lucario Balderas,Manager con equipo y vacantes,True,02/04/2019,092 - ZONA CEDIS MUEBLES IMPORTACION,201854,MX-02-MEX-93 - T11-TECAMAC/ BODEGA VANCOUVER-93,TECAMAC,"TECAMAC, EDO ME",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 EMBARQUES TCMC 03,1053,GERENTE DE EMBARQUES,Manager,GERENTE,33206,GERENTE DE EMBARQUES,238,204,34,8,230,223,77,"[90477306, 90478010, 90477775, 90383218, 90478...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...",[201854],4.95,0.44,5.00,1.47,2.27,6.72,1
109843,94870675,94870675,Maria Teresa Ochoa Gomez,Manager con equipo y vacantes,True,18/11/2022,072 - TRASLADO,218229,MX-05-MEX-40 - R27-OFICINA STAFF-COPPEL-40,TEXCOCO,TEXCOCO EDO MEX,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 MODULO TIENDAS TXCC,39,GERENTE DE AREA CEDIS,Manager,GERENTE,134147,GERENTE DE AREA CEDIS,187,138,49,43,144,185,104,"[90437075, 90473926, 90361942, 90361529, 90474...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[218234, 218229]",5.03,0.37,5.00,1.89,1.99,2.93,2
16021,92336515,92336515,Aaron Aguilar Herrera,Manager con equipo y vacantes,True,24/09/2007,079 - BODEGA ROPA STAFF,218228,MX-05-MEX-40 - R27-OFICINA STAFF-COPPEL-40,TEXCOCO,TEXCOCO EDO MEX,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 RESGUARDO TXCC,39,GERENTE DE AREA CEDIS,Manager,GERENTE,134053,GERENTE DE AREA CEDIS,161,106,55,0,161,127,92,"[90484232, 90483427, 90483469, 90482950, 90484...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[218234, 218228]",4.22,0.54,4.00,1.59,1.16,18.08,2
121084,92268196,92268196,Camilo Leonel Espinosa Barrios,Manager con equipo y vacantes,True,19/06/2008,065 - DESCARGA DISTRIBUCION,153207,MX-04-MEX-74 - R20-TOLC PERSONAL-74,TOLUCA,"TOLUCA, EDO MEX",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 DESCARGA TOLC,1052,GERENTE DE DESCARGA,Manager,GERENTE,19999,GERENTE DE DESCARGA,161,128,33,27,134,155,90,"[90479699, 90368182, 90481489, 90479716, 90425...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...",[153207],4.99,0.40,5.00,1.56,2.16,17.34,1
25648,90156373,90156373,Daniel Martin Lopez Gonzalez,Manager con equipo y vacantes,True,26/02/2021,065 - DESCARGA DISTRIBUCION,169905,MX-01-PUE-989 - T08-AV. MEXICO-989,PUEBLA,"CUAUTLANCINGO, PUE",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 DESCARGA PBLA 02,1052,GERENTE DE DESCARGA,Manager,GERENTE,22291,GERENTE DE DESCARGA,144,93,51,32,112,132,78,"[90479328, 90482889, 90395636, 90483317, 90333...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[169905, 201212]",5.12,0.61,5.00,1.66,2.33,4.65,2


# Métricas Clave para la Modelización de la Rotación de Personal

Las siguientes variables cuantifican aspectos de la **estructura** y **gestión de recursos** del equipo, sirviendo como métricas continuas y relativas esenciales para predecir la probabilidad de rotación del personal subordinado.

***

### 1. Carga de Personas por Manager ($T$)

| Nombre de la Variable | Descripción |
| :--- | :--- |
| **Tamaño del Equipo** | Representa la **cardinalidad** del conjunto de **subordinados directos** del manager, incluyendo las **vacantes disponibles** en el equipo. |

$$T = \text{Número de Subordinados Directos} + \text{Número de Vacantes Disponibles}$$

***

### 2. Índice de Restricción en Contratación ($I$)

| Nombre de la Variable | Descripción |
| :--- | :--- |
| **Índice de Rigidez de Contratación** | Mide la **carga administrativa** e indica el nivel de **dificultad o restricción** para cubrir posiciones. Es la razón entre el total de contrataciones **congeladas** ($C$) y el total de contrataciones **posibles** ($A$), con un ajuste para asegurar la continuidad del cálculo. |

**Fórmula:**
$$I = \frac{C}{A + 1}$$

**Ejemplo Formal:**
Si un manager tiene **3 posiciones congeladas** ($C=3$) y **4 posiciones posibles** ($A=4$), el Índice de Restricción en Contratación es:
$$I = \frac{3}{4 + 1} = \frac{3}{5} = 0.6$$

***

### 3. Disponibilidad de Posiciones ($P_{v}$)

| Nombre de la Variable | Descripción |
| :--- | :--- |
| **Porcentaje de Posiciones Vacantes Presupuestadas** | Mide la **disponibilidad de posiciones** o la proporción de la capacidad no utilizada. Es la razón, expresada como porcentaje, entre el número de **posiciones vacantes** ($V$) y el **total de posiciones a cargo** (presupuestadas, $P$) del manager. |

**Fórmula:**
$$P_{v} = \begin{cases} \frac{V}{P} \times 100 & \text{si } P > 0 \\ 0 & \text{si } P = 0 \end{cases}$$

**Ejemplo Formal:**
Si el **total de posiciones a cargo** es 20 ($P=20$) y el número de **posiciones vacantes** es 4 ($V=4$), el Porcentaje de Posiciones Vacantes Presupuestadas es:
$$P_{v} = \frac{4}{20} \times 100 = 20\%$$

In [105]:
def create_team_structure_features(df):
    """
    Crea variables de estructura, estabilidad y carga de trabajo del manager
    para predecir rotación de subordinados.
    
    Parameters:
    -----------
    df : DataFrame con las columnas necesarias por manager
    
    Returns:
    --------
    DataFrame con nuevas variables agregadas
    """
    
    df_features = df.copy()
    
    # 1. CARGA DE PERSONAS POR MANAGER (Tamaño del Equipo)
    df_features['tamano_equipo'] = df_features['total_plantilla_activa'] + df_features['posiciones_vacantes_activas']
    
        
    # 2. CARGA ADMINISTRATIVA (Índice de Rigidez de Contratación)
    df_features['indice_rigidez_contratacion'] = (
        df_features['total_contrataciones_congeladas'] / 
        (df_features['posiciones_vacantes'] + 1)
    )
    
    # 3. DISPONIBILIDAD DE POSICIONES (% Posiciones Vacantes activas)
    df_features['pct_posiciones_vacantes'] = np.where(
        df_features['tamano_equipo'] > 0,
        (df_features['posiciones_vacantes_activas'] / df_features['tamano_equipo']) * 100,
        0
    )    
    return df_features

In [106]:
df_features = create_team_structure_features(df_jerarquia)
df_features.head()

,numero_de_persona,numero_de_trabajador,nombre_de_colaborador,tipo_rol,tiene_subordinados,fecha_antiguedad_grupo,centro_de_costos,centro,ubicacion,region,ciudad,empleador_legal,unidad_de_negocio,division,departamento,codigo_de_puesto,nombre_de_puesto,familia_de_puesto,funcion_del_puesto,codigo_de_posicion,nombre_de_posicion,total_plantilla_autorizada,total_plantilla_activa,posiciones_vacantes,total_contrataciones_congeladas,total_contrataciones_aprobadas,total_posiciones_presupuestadas,total_indefinidos_a_cargo,lista_subordinados_trabajador,grupos_salariales,lista_centros_subordinados,promedio_grupo_salarial,std_grupo_salarial,moda_grupo_salarial,promedio_antiguedad_subordinados_anos,std_antiguedad_equipo,antiguedad_anois,numero_centros_a_cargo,posiciones_vacantes_activas,tamano_equipo,indice_rigidez_contratacion,pct_posiciones_vacantes
31833,90021474,90021474,Christian Lucario Balderas,Manager con equipo y vacantes,True,02/04/2019,092 - ZONA CEDIS MUEBLES IMPORTACION,201854,MX-02-MEX-93 - T11-TECAMAC/ BODEGA VANCOUVER-93,TECAMAC,"TECAMAC, EDO ME",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 EMBARQUES TCMC 03,1053,GERENTE DE EMBARQUES,Manager,GERENTE,33206,GERENTE DE EMBARQUES,238,204,34,8,230,223,77,"[90477306, 90478010, 90477775, 90383218, 90478...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...",[201854],4.95,0.44,5.00,1.47,2.27,6.72,1,26,230,0.23,11.30
109843,94870675,94870675,Maria Teresa Ochoa Gomez,Manager con equipo y vacantes,True,18/11/2022,072 - TRASLADO,218229,MX-05-MEX-40 - R27-OFICINA STAFF-COPPEL-40,TEXCOCO,TEXCOCO EDO MEX,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 MODULO TIENDAS TXCC,39,GERENTE DE AREA CEDIS,Manager,GERENTE,134147,GERENTE DE AREA CEDIS,187,138,49,43,144,185,104,"[90437075, 90473926, 90361942, 90361529, 90474...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[218234, 218229]",5.03,0.37,5.00,1.89,1.99,2.93,2,6,144,0.86,4.17
16021,92336515,92336515,Aaron Aguilar Herrera,Manager con equipo y vacantes,True,24/09/2007,079 - BODEGA ROPA STAFF,218228,MX-05-MEX-40 - R27-OFICINA STAFF-COPPEL-40,TEXCOCO,TEXCOCO EDO MEX,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 RESGUARDO TXCC,39,GERENTE DE AREA CEDIS,Manager,GERENTE,134053,GERENTE DE AREA CEDIS,161,106,55,0,161,127,92,"[90484232, 90483427, 90483469, 90482950, 90484...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[218234, 218228]",4.22,0.54,4.00,1.59,1.16,18.08,2,55,161,0.00,34.16
121084,92268196,92268196,Camilo Leonel Espinosa Barrios,Manager con equipo y vacantes,True,19/06/2008,065 - DESCARGA DISTRIBUCION,153207,MX-04-MEX-74 - R20-TOLC PERSONAL-74,TOLUCA,"TOLUCA, EDO MEX",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 DESCARGA TOLC,1052,GERENTE DE DESCARGA,Manager,GERENTE,19999,GERENTE DE DESCARGA,161,128,33,27,134,155,90,"[90479699, 90368182, 90481489, 90479716, 90425...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...",[153207],4.99,0.40,5.00,1.56,2.16,17.34,1,6,134,0.79,4.48
25648,90156373,90156373,Daniel Martin Lopez Gonzalez,Manager con equipo y vacantes,True,26/02/2021,065 - DESCARGA DISTRIBUCION,169905,MX-01-PUE-989 - T08-AV. MEXICO-989,PUEBLA,"CUAUTLANCINGO, PUE",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 DESCARGA PBLA 02,1052,GERENTE DE DESCARGA,Manager,GERENTE,22291,GERENTE DE DESCARGA,144,93,51,32,112,132,78,"[90479328, 90482889, 90395636, 90483317, 90333...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[169905, 201212]",5.12,0.61,5.00,1.66,2.33,4.65,2,19,112,0.62,16.96


### Dataset de Encuesta Gerencial

In [107]:
# Cargar dataset de encuesta
df_encuesta = pd.read_excel('../data/Sondeo  -   Perspectiva del Gerente sobre Rotación  (Respuestas).xlsx')

print(f"Dimensiones del dataset: {df_encuesta.shape}")
print(f"Filas: {df_encuesta.shape[0]:,}")
print(f"Columnas: {df_encuesta.shape[1]}")

Dimensiones del dataset: (4997, 9)
Filas: 4,997
Columnas: 9


## Duplicados en encuestas

In [108]:
# Contar cuántas veces aparece cada 'num_empleado'
conteo_duplicados = df_encuesta['num_empleado'].value_counts()

# Convertir el resultado a un DataFrame (opcional, para una mejor visualización)
df_conteo = conteo_duplicados.reset_index()
df_conteo.columns = ['num_empleado', 'veces_repetido']

# Mostrar solo los que se repiten (los duplicados)
duplicados_repetidos = df_conteo[df_conteo['veces_repetido'] > 1]
print(f"Total de duplicados: {duplicados_repetidos['veces_repetido'].sum()}")
print(duplicados_repetidos)

Total de duplicados: 439
     num_empleado  veces_repetido
0        90269173               8
1        90380220               4
2        90011569               3
3        97133663               3
4        96094117               3
..            ...             ...
206      90013966               2
207      98940181               2
208      90166280               2
209      90459419               2
210      96432624               2

[211 rows x 2 columns]


In [109]:
df_encuesta[df_encuesta['num_empleado']== 90269173]

,tiempo,num_empleado,escolaridad,P1,P2,P3,P4,P5,P6
2966,2025-09-09 15:52:21.787,90269173,Licenciatura o Ingeniería,Prestaciones y hambiente laboral,No se adaptan al cambio y la exigencia de algu...,Cambio en los incentivos ganan menos incentivo...,La idea negativa de algunos colaboradores,Platicas de seguimiento y una explicación corr...,No
4151,2025-09-11 18:25:07.709,90269173,Licenciatura o Ingeniería,Las prestaciones y el ambiente laboral,No se adaptan al cambio no habían trabajado co...,Cambian su incentivo y al final de mes ganan m...,"Edades del colaborador, la ideología que llega...",Preguntar desde inicio si han trabajado con metas,No
4830,2025-09-15 15:09:02.734,90269173,Licenciatura o Ingeniería,Prestaciones y hambiente laboral,No se adaptan a los cambios y metas,Cambio en sus ingresos con los nuevos unsentivos,Colaboradores con otra ideología y experiencia...,Que conozcan a detalle las prestaciones,No
4831,2025-09-15 15:13:06.964,90269173,Licenciatura o Ingeniería,Las prestaciones y ambiente laboral,No se adaptan a coppel o a las metas,Cambio en sus ingresos en los insentivos,Ideología de colaboradores con varios años en ...,Que conozcan todas las prestaciones los colabo...,No
4832,2025-09-15 15:18:15.325,90269173,Licenciatura o Ingeniería,Las prestaciones y ambiente laboral,No se adaptan al cambio y atrabajar sobre metas,"El cambio en sus ingresos, reciben menos incen...",La ideología de los colaboradores con más tiempo,La dec. De prestaciones,No
4833,2025-09-15 15:23:23.886,90269173,Licenciatura o Ingeniería,Las prestaciones y ambiente laboral,No se adaptan a coppel y a las metas,El cambio en sus ingresos menos incentivo,Ideología de los colaboradores con mucho tiemp...,Dec de las prestación o beneficios coppel,No
4861,2025-09-15 17:41:20.668,90269173,Licenciatura o Ingeniería,Las prestaciones y ambiente laboral,No sé adaptan al cambio y a las metas,"El cambio en sus ingresos, reciben menos incen...",La ideología de varios colaboradores que ya ll...,La Dec. Prestaciones o beneficios coppel,No
4863,2025-09-15 17:50:02.571,90269173,Licenciatura o Ingeniería,Las prestaciones y el ambiente laboral,No sé adaptan al cambio y no habían trabajado ...,Cambio en sus ingresos,Colaboradores que llevan varios años en la emp...,La Dec. Beneficios Coppel,No


In [110]:
# Información del dataset
df_encuesta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4997 entries, 0 to 4996
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   tiempo        4997 non-null   datetime64[ns]
 1   num_empleado  4997 non-null   int64         
 2   escolaridad   4997 non-null   object        
 3   P1            4996 non-null   object        
 4   P2            4996 non-null   object        
 5   P3            4981 non-null   object        
 6   P4            4993 non-null   object        
 7   P5            4993 non-null   object        
 8   P6            4968 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(7)
memory usage: 351.5+ KB


In [111]:
df_encuesta = df_encuesta[['num_empleado', 'escolaridad']]
filas_antes = len(df_encuesta)
df_encuesta = df_encuesta.drop_duplicates(subset=['num_empleado'], keep='first')

filas_despues = len(df_encuesta)
diferencia = filas_antes - filas_despues

print(f"El DataFrame original tenía {filas_antes} filas.")
print(f"El DataFrame sin duplicados tiene {filas_despues} filas.")
print(f"Se eliminaron {diferencia} filas duplicadas.")

El DataFrame original tenía 4997 filas.
El DataFrame sin duplicados tiene 4769 filas.
Se eliminaron 228 filas duplicadas.


## Integración de Datasets

In [112]:
df_entrevistados_rotacion = df_encuesta.merge(
    df_features,
    left_on='num_empleado',
    right_on='numero_de_trabajador',
    how='inner'
)
print(f'Dimensiones del dataset final: {df_entrevistados_rotacion.shape[0]} anteriormente {df_encuesta.shape[0]}')
print(f'Se perdieron: {df_encuesta.shape[0] - df_entrevistados_rotacion.shape[0]} registros') 


Dimensiones del dataset final: 4670 anteriormente 4769
Se perdieron: 99 registros


In [113]:
df_entrevistados_rotacion.to_csv('../data/df_entrevistados_rotacion.csv', index=False)
df_entrevistados_rotacion.head()

,num_empleado,escolaridad,numero_de_persona,numero_de_trabajador,nombre_de_colaborador,tipo_rol,tiene_subordinados,fecha_antiguedad_grupo,centro_de_costos,centro,ubicacion,region,ciudad,empleador_legal,unidad_de_negocio,division,departamento,codigo_de_puesto,nombre_de_puesto,familia_de_puesto,funcion_del_puesto,codigo_de_posicion,nombre_de_posicion,total_plantilla_autorizada,total_plantilla_activa,posiciones_vacantes,total_contrataciones_congeladas,total_contrataciones_aprobadas,total_posiciones_presupuestadas,total_indefinidos_a_cargo,lista_subordinados_trabajador,grupos_salariales,lista_centros_subordinados,promedio_grupo_salarial,std_grupo_salarial,moda_grupo_salarial,promedio_antiguedad_subordinados_anos,std_antiguedad_equipo,antiguedad_anois,numero_centros_a_cargo,posiciones_vacantes_activas,tamano_equipo,indice_rigidez_contratacion,pct_posiciones_vacantes
0,90097812,Licenciatura o Ingeniería,90097812,90097812,Fernando Toledo Lara,Manager con equipo y vacantes,True,08/07/2020,007 - COBRANZAS,503101,MX-03-NAY-226 - C01-SANTIAGO IXCUINTLA-226,GUADALAJARA,"SANTIAGO, IXC",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 OPERACION DE COBRANZA GDLJ I 04,721,GERENTE DE OPERACION COBRANZA,Manager,GERENTE,7774,GERENTE DE OPERACION COBRANZA,31,29,2,0,31,29,30,"[90475337, 90484419, 90435830, 90345222, 90373...","[8, 5, 5, 5, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, ...",[503101],5.65,1.38,5.00,1.10,1.55,5.21,1,2,31,0.00,6.45
1,92650554,Licenciatura o Ingeniería,92650554,92650554,Ulises Ruiz Aguilar,Manager con equipo y vacantes,True,16/04/2008,001 - TDAS MUEBLES,785701,MX-03-MEX-149 - C04-ECATEPEC DE MORELOS III-149,TECAMAC,ECATEPEC DE MOR,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 TIENDA MUEBLES 7857 LAS BRISAS,1031,GERENTE DE TIENDA MUEBLES,Manager,GERENTE,140562,GERENTE DE TIENDA MUEBLES,20,10,10,2,18,18,12,"[90304458, 90392225, 90267723, 90273641, 90021...","[6, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 4, ...",[785701],4.85,0.49,5.00,2.81,1.56,17.52,1,8,18,0.18,44.44
2,92109391,Preparatoria,92109391,92109391,Elizabeth Arlin Jimenez Morales,Manager con equipo y vacantes,True,22/10/2009,007 - COBRANZAS,501102,MX-03-BC-62 - C11-MEXICALI II-62,MEXICALI,"MEXICALI, BCN.",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 OPERACION DE COBRANZA MXLI 02,721,GERENTE DE OPERACION COBRANZA,Manager,GERENTE,6595,GERENTE DE OPERACION COBRANZA,50,46,4,0,50,48,50,"[90474702, 90479988, 90313608, 90355707, 90380...","[8, 5, 5, 5, 5, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, ...",[501102],5.86,1.62,5.00,2.04,2.49,16.00,1,4,50,0.00,8.00
3,90391277,Licenciatura o Ingeniería,100021966,90391277,Rusia Muñiz Fuentes,Manager con equipo y vacantes,True,02/08/2024,007 - COBRANZAS,546501,MX-03-CHIH-9 - C05-ASCENCION-9,CD JUAREZ,"ASCENSION, CHIH",COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 OPERACION DE COBRANZA CDJZ 14,721,GERENTE DE OPERACION COBRANZA,Manager,GERENTE,155461,GERENTE DE OPERACION COBRANZA,4,4,0,0,4,4,4,"[98720937, 90119540, 90461481, 90472439]","[6, 5, 5, 5]",[546501],5.25,0.50,5.00,1.49,2.37,1.71,1,0,4,0.00,0.00
4,90269515,Licenciatura o Ingeniería,90269515,90269515,Gustavo Alexis Talavera Chavez,Manager con equipo y vacantes,True,11/10/2022,005 - TDAS CANADAS,117102,MX-01-MICH-1073 - T19-SILVA-1073,TOLUCA,ARIO DE ROSALES,COPPEL SA DE CV,BU_CSACV,DIRECCION DE OPERACIONES DE GC,07 TIENDA ROPA 1171 SILVA,1032,GERENTE DE TIENDA ROPA,Manager,GERENTE,85127,GERENTE DE TIENDA ROPA,9,5,4,2,7,8,6,"[96657367, 90179108, 90228504, 90264295, 90231...","[5, 5, 5, 5, 5, 5, 4, 5, 5]",[117102],4.89,0.33,5.00,5.30,3.70,2.95,1,2,7,0.40,28.57


---

## Fin de Integracion de Datasets

**Autor**: José Eduardo Rodriguez Barrios 

**Versión**: 1.0  

**Herramientas**: Python 3.11, pandas, numpy, matplotlib, seaborn

---

In [127]:
df_entrevistados_rotacion.columns

Index(['num_empleado', 'escolaridad', 'numero_de_persona',
       'numero_de_trabajador', 'nombre_de_colaborador', 'tipo_rol',
       'tiene_subordinados', 'fecha_antiguedad_grupo', 'centro_de_costos',
       'centro', 'ubicacion', 'region', 'ciudad', 'empleador_legal',
       'unidad_de_negocio', 'division', 'departamento', 'codigo_de_puesto',
       'nombre_de_puesto', 'familia_de_puesto', 'funcion_del_puesto',
       'codigo_de_posicion', 'nombre_de_posicion',
       'total_plantilla_autorizada', 'total_plantilla_activa',
       'posiciones_vacantes', 'total_contrataciones_congeladas',
       'total_contrataciones_aprobadas', 'total_posiciones_presupuestadas',
       'total_indefinidos_a_cargo', 'lista_subordinados_trabajador',
       'grupos_salariales', 'lista_centros_subordinados',
       'promedio_grupo_salarial', 'std_grupo_salarial', 'moda_grupo_salarial',
       'promedio_antiguedad_subordinados_anos', 'std_antiguedad_equipo',
       'antiguedad_anois', 'numero_centros_a_carg